In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [ ]:
# MAGIC %run /Workspace/Users/pawanvirat32@gmail.com/consolidated_pipeline/1_setup/utilities

In [ ]:
print(bronze_schema, silver_schema, gold_schema)

In [ ]:
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'customers', 'Data Source')

In [ ]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

print(f"catalog: {catalog}")
print(f"data_source: {data_source}")

In [ ]:
base_path = f"s3://sportsbar-pr1/{data_source}/*csv"
print(base_path)

In [ ]:
df =(
    spark.read.format('csv')
    .option('header', 'true')
    .option('inferSchema', 'true')
    .load(base_path)
    .withColumn('read_timestamp', F.current_timestamp())
    .select('*', '_metadata.file_name', '_metadata.file_size')
    )
display(df)

In [ ]:
df.printSchema()

In [ ]:
df.write\
    .format('delta')\
    .option('delta.enableChangeDataFeed', 'true')\
    .mode('overwrite')\
    .saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')

## silver processing

In [ ]:
df_bronze = spark.sql(f'select * from {catalog}.{bronze_schema}.{data_source}')
display(df_bronze)

In [ ]:
df_bronze.printSchema()

In [ ]:
df_duplicates = df_bronze.groupBy('customer_id').count().filter(F.col('count') > 1)
display(df_duplicates)

In [ ]:
print('rows before deleting duplicates: ', df_bronze.count())
df_silver = df_bronze.dropDuplicates(['customer_id'])
print('rows after deleting duplicates: ', df_silver.count())

In [ ]:
df_silver.filter(F.col('customer_name') != F.trim(F.col('customer_name'))).show()

In [ ]:
# removing leading spaces from customer names
df_silver = df_silver.withColumn('customer_name', F.trim(F.col('customer_name')))
display(df_silver.filter(F.col('customer_name') != F.trim(F.col('customer_name'))))

In [ ]:
display(df_silver.select('city').distinct())

In [ ]:
city_mapping = {
    "Bengalore": "Bengaluru",
    "Bengaluruu": "Bengaluru",

    "Hyderabadd": "Hyderabad",
    "Hyderbad": "Hyderabad",

    "NewDelhee": "New Delhi",
    "NewDelhi": "New Delhi",
    "NewDheli": "New Delhi"
}

allowed = ['New Delhi', 'Bengaluru', 'Hyderabad']

df_silver = (
    df_silver
    .replace(city_mapping, subset = ['city'])
    .withColumn(
        'city',
        F.when(F.col('city').isNull(), None)
        .when(F.col('city').isin(allowed), F.col('city'))
        .otherwise(None)))

display(df_silver.select('city').distinct())

In [ ]:
display(df_silver.select('customer_name').distinct())

In [ ]:
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(), None)
     .otherwise(F.initcap(F.col("customer_name")))
)

display(df_silver.select('customer_name').distinct())

In [ ]:
display(df_silver.filter(F.col('city').isNull()))

In [ ]:
null_cust_name = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
display(df_silver.filter(F.col('customer_name').isin(null_cust_name)))

In [ ]:
# city connections confirmed by business team
city_mapping = {
    789403: "New Delhi",
    789420: "Bengaluru",
    789521: "Hyderabad",
    789603: "Hyderabad"
}

In [ ]:
df_fix = spark.createDataFrame(
    [(k, v) for k, v in city_mapping.items()],
    ['customer_id', 'city_fix']
)

display(df_fix)

In [ ]:
df_silver = (
    df_silver
    .join(df_fix, 'customer_id', 'left')
    .withColumn(
        'city',
        F.coalesce('city', 'city_fix')
    )
    .drop('city_fix')
)

display(df_silver)

In [ ]:
display(df_silver.filter(F.col('city').isNull()))

In [ ]:
df_silver = df_silver.withColumn('customer_id', F.col('customer_id').cast('string'))
df_silver.printSchema()

#### our main company's gold contains diff cols, so we cannot merge this layer with gold layer

In [ ]:
df_silver = (
  df_silver
  .withColumn(
    'customer',
    F.concat_ws('-', 'customer_name', F.coalesce(F.col('city'), F.lit('Unknown')))
  )

  # static attributes align with parent data model
  .withColumn('market', F.lit('India'))
  .withColumn('platform', F.lit('Sports Bar'))
  .withColumn('channel', F.lit('Acquisition'))
)


display(df_silver)

In [ ]:
df_silver.write\
  .format('delta')\
  .option('delta.enableChangeDataFeed', 'true')\
  .option('mergeSchema', 'true')\
  .mode('overwrite')\
  .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')

## Gold processing

In [ ]:
df_silver = spark.sql(f'SELECT * FROM {catalog}.{silver_schema}.{data_source};')

# take required columns only
# customer_id, customer_name, city, read_timestamp, file_name, file_size, customer, market, platform, channel
df_gold = df_silver.select('customer_id', 'customer_name', 'city', 'customer', 'market', 'platform', 'channel')

display(df_gold)

In [ ]:
df_gold.write\
  .format('delta')\
  .option('delta.enableChangeDataFeed', 'true')\
  .mode('overwrite')\
  .saveAsTable(f'{catalog}.{gold_schema}.sb_dim_{data_source}')

In [ ]:
delta_table = DeltaTable.forName(spark, f'{catalog}.{gold_schema}.dim_{data_source}')

df_child_customers = spark.table(f'{catalog}.{gold_schema}.sb_dim_{data_source}').select(
    F.col('customer_id').alias('customer_code'),
    'customer',
    'market',
    'platform',
    'channel'
)

In [ ]:
delta_table.alias('target').merge(
    source = df_child_customers.alias('source'),
    condition = 'target.customer_code = source.customer_code'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()